In [1]:
import os
import yaml
import copy
import pickle


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from os import path as pt
from tqdm import tqdm
sns.set_style("darkgrid")  # 원하는 스타일 선택

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import ml_collections
from src.utils import *
from src.evaluation.summary import full_evaluation

### Data Preprocess

In [2]:
def rolling_window(data, window_size):
    n_windows = data.shape[0] - window_size + 1
    windows = np.zeros((n_windows, window_size, data.shape[1]))
    for idx in range(n_windows):
        windows[idx] = data[idx:idx + window_size]
    return windows

In [3]:
# --- [0. 준비 단계] ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 128

# 데이터 로드
df = pd.read_csv(f"./data/indices.csv")
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df.set_index('Date', inplace=True)
df = df.apply(pd.to_numeric).astype(float)

log_returns = np.log1p(df.pct_change()).dropna().values
init_price_raw = np.array(df)[:-1, :]

# --- [1. Early Split (시간 순 분할)] ---
total_len = len(log_returns)
train_size = int(0.7 * total_len)

train_raw = log_returns[:train_size]
test_raw = log_returns[train_size:] # Test는 원본 6차원 그대로 대기

train_init_raw = init_price_raw[:train_size]
test_init_raw = init_price_raw[train_size:]

# --- [2. GARCH 12차원 전처리 (Train ONLY)] ---
# Train만 GARCH로 12차원 분해 및 스케일링 수행
train_12d, scalers = scaling(train_raw)


train_12d = train_12d[:train_size]

# --- [3. Windowing (시퀀스 쪼개기)] ---
# Train은 12차원 윈도우 생성
train_windowed = torch.from_numpy(rolling_window(train_12d, seq_len)).float()
train_init_windowed = torch.from_numpy(train_init_raw[:-(seq_len-1)]).float().unsqueeze(1)

test_windowed = torch.from_numpy(rolling_window(test_raw, seq_len)).float()
test_init_windowed = torch.from_numpy(test_init_raw[:-(seq_len-1)]).float().unsqueeze(1)

# (B, T, C) -> (B, C, T) 축 변환
training_data = train_windowed.clone().detach() # (B, 256, 12)
test_data = test_windowed.clone().detach()  # (B, 256, 6)

training_set = TensorDataset(training_data)
test_set = TensorDataset(test_data)

train_dl = DataLoader(training_set, batch_size=256, shuffle=False)
test_dl = DataLoader(test_set, batch_size=256, shuffle=False)

print("\n🚀 데이터 준비 완료!")
print(f"Train DataLoader 배치 크기: {next(iter(train_dl))[0].shape}")
print(f"Test DataLoader 배치 크기: {next(iter(test_dl))[0].shape}")

c:\Users\Juhwankim\Documents\Github\quantDiffusion\modules\QuantGAN\src\preprocess\gaussianize.py:179: UserWarning: Warning: No convergence after 100 iterations. Increase max_iter.
  warnings.warn("Warning: No convergence after %d iterations. Increase max_iter." % max_iter)



🚀 데이터 준비 완료!
Train DataLoader 배치 크기: torch.Size([256, 128, 6])
Test DataLoader 배치 크기: torch.Size([256, 128, 6])


c:\Users\Juhwankim\Documents\Github\quantDiffusion\modules\QuantGAN\src\preprocess\gaussianize.py:179: UserWarning: Warning: No convergence after 100 iterations. Increase max_iter.
  warnings.warn("Warning: No convergence after %d iterations. Increase max_iter." % max_iter)


### Generative models for time series generation

### Data Construction

We divide the data into training and validation set for the offline evaluation of our model

Load the dataset


In [4]:
config = {
    # Data-related settings
    'file_name': 'indices',
    'data_dir': 'data/',
    'result_dir': 'results/',
    'log_interval': 5,
    'n_vars': 6,
    'n_steps': 128,  # 실제 데이터 시퀀스 길이와 일치 (training_data: 128 스텝)

    # Training settings
    'device': 'cuda',
    'seed': 0,
    'batch_size': 256,
    'n_epochs': 300,

    # Algorithm-related settings
    'algo': 'QuantGAN',

    # Model settings
    'G_hidden_dim': 80,
    'D_hidden_dim': 80,
    'output_dim': 1,
    'noise_dim': 6,
    'clip': 0.01,

    # Learning rates
    'lr_G': 0.0002,
    'lr_D': 0.0001,
    'D_steps_per_G_step': 1
}

from types import SimpleNamespace

# 기존 config 딕셔너리를 객체 형식으로 변환
config = SimpleNamespace(**config)

### Generative model

Here we construct a generator and a discriminator for this task. Both the generator and discriminator takes as input the time series. Then we have the training algorithm TailGANTrainer.

### Initialize the generator, discriminator and the trainer

In [5]:
from src.baselines.networks.discriminators import UserDiscriminator
from src.baselines.networks.generators import UserGenerator
from src.baselines.trainer import *

generator = UserGenerator(config)
discriminator = UserDiscriminator(config)
trainer = GANTrainer(G=generator, D=discriminator, train_dl=train_dl, config=config)

In [6]:
def count_parameters(model):
    # 학습 가능한(Trainable) 파라미터 수
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 전체(Total) 파라미터 수
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"✅ 전체 파라미터 수: {total_params:,}")
    print(f"✅ 학습 가능한 파라미터 수: {trainable_params:,}")
    return total_params

# 모델 인스턴스를 전달하여 실행
count_parameters(generator)

✅ 전체 파라미터 수: 165,046
✅ 학습 가능한 파라미터 수: 165,046


165046

### Model training and saving

In [7]:
# Model training
trainer.fit()

  0%|          | 0/300 [00:00<?, ?it/s]

Batch 0: D Loss: 3.706317663192749, Generator Loss: 0.6879758834838867
Batch 5: D Loss: 1.3864130973815918, Generator Loss: 0.6882537007331848
Batch 10: D Loss: 1.3864827156066895, Generator Loss: 0.6882634162902832


  0%|          | 1/300 [00:03<16:54,  3.39s/it]

Batch 15: D Loss: 1.3865392208099365, Generator Loss: 0.6882256269454956
Batch 0: D Loss: 1.3865041732788086, Generator Loss: 0.6882363557815552
Batch 5: D Loss: 1.3865209817886353, Generator Loss: 0.6882648468017578
Batch 10: D Loss: 1.3864696025848389, Generator Loss: 0.6883128881454468


  1%|          | 2/300 [00:05<12:49,  2.58s/it]

Batch 15: D Loss: 1.3863826990127563, Generator Loss: 0.6883864998817444
Batch 0: D Loss: 1.386345624923706, Generator Loss: 0.6884210109710693
Batch 5: D Loss: 1.3862547874450684, Generator Loss: 0.6884750723838806
Batch 10: D Loss: 1.3863195180892944, Generator Loss: 0.688555121421814


  1%|          | 3/300 [00:07<11:24,  2.31s/it]

Batch 15: D Loss: 1.3861379623413086, Generator Loss: 0.6886377930641174
Batch 0: D Loss: 1.3862357139587402, Generator Loss: 0.6886013746261597
Batch 5: D Loss: 1.3862050771713257, Generator Loss: 0.688645601272583


  1%|          | 3/300 [00:08<14:15,  2.88s/it]


KeyboardInterrupt: 

# Model Evaluation

In [9]:
from scipy.stats import entropy, wasserstein_distance, ks_2samp, spearmanr, kendalltau
from statsmodels.tsa.stattools import acf
from scipy.spatial.distance import jensenshannon

full_name = "indices_300_256_G_lr_0.0002_D_lr_0.0001_noise_6_hidden_dim_80"

def compute_avg_emd(real_data, fake_data, window):
    """
    Compute the Earth Mover's Distance (EMD) for real and fake data over a rolling window.
    """
    emd = 0
    for i in range(real_data.shape[1]):  # Iterate over features
        real_dist = rolling_window(real_data[:, i, :].T, window).sum(axis=1).ravel()
        fake_dist = rolling_window(fake_data[:, i, :].T, window).sum(axis=1).ravel()
        emd += wasserstein_distance(real_dist, fake_dist)
    return emd

window = 100

In [11]:
""" Find the best epochs based on 100 days cumulative distribution """

gen_tmp = UserGenerator(config)
min_dist = float('inf')  
best_epoch = 0

batch_size = 1000
for epoch in range(50, 300, 5):
    
    # Load generator for the current epoch
    generator.load_state_dict(torch.load(f'./results/models/{full_name}/Generator_{epoch}.pt'))
    generator.to(config.device)
    generator.eval()
    
    # Generate fake data
    noise = torch.randn(batch_size, config.noise_dim, config.n_steps, device=config.device)
    with torch.no_grad():        
        fake= generator(noise)                
        
    # Inverse scaling for real and fake data
    fake_data = inverse_scaling(fake, scalers)        
    real_data = inverse_scaling(training_data.transpose(1, 2), scalers)
    
    # Compute EMD
    emd = compute_avg_emd(real_data, fake_data, window)

    # Update best epoch if current EMD is lower
    if emd < min_dist:
        min_dist = emd
        best_epoch = epoch
        print(f"min_distance: {min_dist:.3f}, best_epoch: {best_epoch}") 

generator.load_state_dict(torch.load(f'./results/models/{full_name}/Generator_{best_epoch}.pt'))
generator.to(config.device)
generator.eval()
print()

min_distance: 178.027, best_epoch: 50
min_distance: 63.015, best_epoch: 60
min_distance: 0.854, best_epoch: 90
min_distance: 0.423, best_epoch: 105
min_distance: 0.411, best_epoch: 110
min_distance: 0.401, best_epoch: 115
min_distance: 0.395, best_epoch: 120
min_distance: 0.347, best_epoch: 130
min_distance: 0.201, best_epoch: 150


KeyboardInterrupt: 

In [ ]:
generator.load_state_dict(torch.load(f'./results/models/{full_name}/Generator_150.pt'))
""" Fake data와 Test data를 비교 """
batch_size = 2000
noise = torch.randn(batch_size, config.noise_dim, config.n_steps, device=config.device)
with torch.no_grad():
    fake = generator(noise)

fake_data = inverse_scaling(fake, scalers)
real_data = inverse_scaling(training_data.transpose(1, 2), scalers)
emd = compute_avg_emd(real_data, fake_data, window)
print(emd)

0.20023713392618564


In [10]:
def rolling_window(data, window_size):
    n_windows = data.shape[0] - window_size + 1
    windows = np.zeros((n_windows, window_size, data.shape[1]))
    for idx in range(n_windows):
        windows[idx] = data[idx:idx + window_size]
    return windows

In [11]:
import gc
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm, kstest # KS 검정을 위해 kstest 추가

from statsmodels.tsa.stattools import acf
from scipy.stats import skew, kurtosis, wasserstein_distance, ks_2samp




def rolling_window(data, window_size):
    n_windows = data.shape[0] - window_size + 1
    windows = np.zeros((n_windows, window_size, data.shape[1]))
    for idx in range(n_windows):
        windows[idx] = data[idx:idx + window_size]
    return windows


def _scalar_diff_loss(diff, loss_type):
    """Compute scalar loss from diff array: mse, mae, or frobenius."""
    if loss_type == 'mse':
        return np.mean(diff ** 2)
    if loss_type == 'mae':
        return np.mean(np.abs(diff))
    if loss_type in ('frobenius', 'fro'):
        return np.linalg.norm(diff)
    raise ValueError(f"Unsupported loss_type: {loss_type}")


def correlation_mean(data):
    """
    각 샘플의 상관행렬을 계산한 후, 그 평균을 반환.
    
    Parameters:
        data (np.ndarray): shape = (num_samples, num_time_steps, num_features)
    
    Returns:
        mean_correlation_matrix (np.ndarray): shape = (num_features, num_features)
    """
    num_samples = data.shape[0]
    correlations = []

    for i in range(num_samples):
        sample = data[i]  # shape = (num_time_steps, num_features)
        corr_mat = np.corrcoef(sample, rowvar=False)
        correlations.append(corr_mat)

    mean_correlation_matrix = np.mean(correlations, axis=0)  
    return mean_correlation_matrix


def ccf_mean_vectorized(data, max_lag=10):
    """
    행렬 연산(Vectorization)을 활용하여 매우 빠르고 메모리 효율적인 CCF 평균 계산
    - data: shape = (num_samples, time_steps, num_features)
    """
    B, T, F = data.shape

    # 1. 시간 축(axis=1)을 기준으로 전체 평균과 표준편차 계산
    mu = np.mean(data, axis=1, keepdims=True)  # (B, 1, F)
    # 모집단 표준편차(ddof=0) 사용: statsmodels의 acovf 기본값과 일치
    std = np.std(data, axis=1, keepdims=True)  # (B, 1, F)

    # 2. Z-Score 정규화 (표준편차가 0인 경우 0으로 나누는 것을 방지하기 위해 np.divide 사용)
    Z = np.divide(data - mu, std, out=np.zeros_like(data), where=std != 0)

    # 3. 유효한(valid) 데이터 마스크 생성 (std가 0이 아닌 정상 샘플 필터링)
    valid_mask = (std[:, 0, :] != 0)  # (B, F)
    # i번째 feature와 j번째 feature 둘 다 유효해야 하므로 (B, F, F) 마스크 생성
    valid_pair_mask = valid_mask[:, :, None] & valid_mask[:, None, :]  

    # 4. 결과를 저장할 배열: (F, F, max_lag + 1)
    mean_ccf = np.zeros((F, F, max_lag + 1))

    # 5. 각 시차(lag)별로 행렬곱 연산 수행 (for문은 단 max_lag+1 번만 돎)
    for k in range(max_lag + 1):
        if k == 0:
            Z_0 = Z
            Z_k = Z
        else:
            Z_0 = Z[:, :-k, :]  # 0 ~ (T-k-1)
            Z_k = Z[:, k:, :]   # k ~ (T-1)

        # einsum을 이용해 배치(b) 및 시간(t) 축에 대해 3차원 행렬을 한 번에 내적(Dot product)
        # Z_0: (B, T-k, F), Z_k: (B, T-k, F) -> 결과: (B, F, F)
        # statsmodels.ccf(adjusted=False)의 공식에 맞춰 분모를 전체 길이 T로 나눔
        ccf_k_batch = np.einsum('bti, btj -> bij', Z_0, Z_k) / T

        # 비정상 샘플(표준편차 0이라 연산 불가)의 결과는 제외하기 위해 마스킹 처리
        ccf_k_batch_valid = np.where(valid_pair_mask, ccf_k_batch, 0)

        # 배치 차원(axis=0)에 대해 합산 및 유효 개수 카운트
        sum_ccf = np.sum(ccf_k_batch_valid, axis=0)    # (F, F)
        count_valid = np.sum(valid_pair_mask, axis=0)  # (F, F)

        # 평균 계산 (유효 샘플이 0개인 구간은 NaN 방지를 위해 0으로 채움)
        mean_ccf[:, :, k] = np.divide(sum_ccf, count_valid, out=np.zeros_like(sum_ccf, dtype=float), where=count_valid != 0)

    return mean_ccf


def _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='mse'):
    """
    (기존 사용하시던 로직 유지. _scalar_diff_loss 함수가 정의되어 있다고 가정합니다.)
    """
    diff = fake_mean_ccf - real_mean_ccf
    # 사용자의 환경에 맞게 오차 계산 (아래는 예시)
    if loss_type == 'mae':
        loss_val = np.mean(np.abs(diff))
    elif loss_type == 'mse':
        loss_val = np.mean(diff ** 2)
    elif loss_type == 'frobenius':
        loss_val = np.linalg.norm(diff)
    else:
        # 기존에 _scalar_diff_loss를 호출하셨다면 해당 함수 사용
        loss_val = _scalar_diff_loss(diff, loss_type) 
        
    return round(float(loss_val), 4)


def calculate_ccf(real_data, fake_data, max_lag):
    CCF_dict = {}

    # CCF 계산을 위해 데이터 전치 후, 백터화 함수 직접 호출
    real_mean_ccf = ccf_mean_vectorized(np.transpose(real_data, (0, 2, 1)), max_lag=max_lag)
    fake_mean_ccf = ccf_mean_vectorized(np.transpose(fake_data, (0, 2, 1)), max_lag=max_lag)

    # 손실 계산
    loss_mae = _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='mae')
    loss_frob = _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='frobenius')
    loss_mse = _scaled_ccf_loss(real_mean_ccf, fake_mean_ccf, loss_type='mse')

    if max_lag == 0:
        CCF_dict[f'CCF(τ = 0)_MAE'] = loss_mae
        CCF_dict[f'CCF(τ = 0)_FROB'] = loss_frob
        CCF_dict[f'CCF(τ = 0)_MSE'] = loss_mse
    
    else:
        CCF_dict[f'CCF(τ > 0)_MAE'] = loss_mae
        CCF_dict[f'CCF(τ > 0)_FROB'] = loss_frob
        CCF_dict[f'CCF(τ > 0)_MSE'] = loss_mse

    df_avg_scores = pd.DataFrame(list(CCF_dict.values()), index=CCF_dict.keys(), columns=['Average'])
    df_avg_scores.index.name = 'Metric'

    return df_avg_scores


def calculate_distribution_scores(real, fake, windows):
    """
    real: (B_real, companys, T)
    fake: (B_fake, companys, T)
    """
    # 1. 배열 차원 확보 (2D가 들어와도 3D로 안전하게 처리)
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    # B는 다를 수 있으므로 각각 추출, companys(C)와 T는 동일하다고 가정
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape
    
    results = {}
    metrics = ['EMD', 'KS']
    
    for w in windows:
        # 1. O(1) 롤링 윈도우 연산 (시간 축인 axis=2 기준 누적합)
        real_cs = np.insert(np.cumsum(real, axis=2), 0, 0, axis=2)
        fake_cs = np.insert(np.cumsum(fake, axis=2), 0, 0, axis=2)
        
        real_roll = real_cs[:, :, w:] - real_cs[:, :, :-w]  # (B_r, C, T - w + 1)
        fake_roll = fake_cs[:, :, w:] - fake_cs[:, :, :-w]  # (B_f, C, T - w + 1)
        
        # ==========================================================
        # 3. EMD & KS Statistic 연산 (종목별 풀링 분포 거리 측정)
        # ==========================================================
        # B가 다르므로 1:1 매칭 불가 -> 종목(C)별로 모든 배치의 데이터를 하나로 풀링
        # np.swapaxes를 통해 (C, B, T_w)로 바꾼 뒤 평탄화 -> Shape: (C, B * T_w)
        real_pooled = np.swapaxes(real_roll, 0, 1).reshape(C, -1)
        fake_pooled = np.swapaxes(fake_roll, 0, 1).reshape(C, -1)
        
        emd_scores = np.zeros(C)
        ks_scores = np.zeros(C)
        
        # 각 종목(Company) 단위로 완성된 두 분포의 거리를 측정 (for문 C번만 실행됨)
        for c in range(C):
            emd_scores[c] = wasserstein_distance(real_pooled[c], fake_pooled[c])
            ks_scores[c], _ = ks_2samp(real_pooled[c], fake_pooled[c])
            
        # ==========================================================
        # 4. 결과 저장
        # ==========================================================
        results[f'EMD({w})'] = np.round(np.mean(emd_scores), decimals=4)
        results[f'KS({w})'] = np.round(np.mean(ks_scores), decimals=4)
    # DataFrame 포맷팅
    ordered_keys = [f"{m}({w})" for m in metrics for w in windows]
    df_avg_scores = pd.DataFrame([results[k] for k in ordered_keys], index=ordered_keys, columns=['Average'])
    df_avg_scores.index.name = 'Metric'
    
    return df_avg_scores

def calculate_skew_kurtosis(real_data, fake_data, windows):
    """
    각 샘플(배치)별 Skew/Kurt를 계산하여 평균 차이를 구하고, 
    요청하신 단일 컬럼 DataFrame 형식으로 반환합니다.
    - real_data: (B_real, C, T)
    - fake_data: (B_fake, C, T)
    """
    B_r, C, T_r = real_data.shape
    B_f, _, T_f = fake_data.shape
    
    # 내부에서 자산별 리스트 생성 (제시해주신 입력 방식 반영)
    real_list = [real_data[:, i, :] for i in range(C)]
    fake_list = [fake_data[:, i, :] for i in range(C)]
    
    results = {}
    metrics = ['Skew', 'Kurt'] # EMD, KS는 외부에서 합친다고 가정하거나 여기서는 Skew/Kurt만 처리

    for w in windows:
        skew_diffs_assets = []
        kurt_diffs_assets = []

        for i in range(C):
            real_arr = np.asarray(real_list[i])   # (B_r, T)
            fake_arr = np.asarray(fake_list[i])   # (B_f, T)

            # 배치 사이즈가 다를 경우 최소값에 맞춤
            num = min(real_arr.shape[0], fake_arr.shape[0])
            r_arr = real_arr[:num]
            f_arr = fake_arr[:num]

            # 롤링 윈도우 합산 (CumSum 활용 최적화)
            r_cs = np.pad(np.cumsum(r_arr, axis=1), ((0, 0), (1, 0)), mode="constant")
            f_cs = np.pad(np.cumsum(f_arr, axis=1), ((0, 0), (1, 0)), mode="constant")

            r_roll = r_cs[:, w:] - r_cs[:, :-w]
            f_roll = f_cs[:, w:] - f_cs[:, :-w]

            # 배치별 skew / kurtosis 계산 (axis=1)
            r_skews = skew(r_roll, axis=1, bias=True, nan_policy="omit")
            f_skews = skew(f_roll, axis=1, bias=True, nan_policy="omit")

            r_kurts = kurtosis(r_roll, axis=1, fisher=False, bias=True, nan_policy="omit")
            f_kurts = kurtosis(f_roll, axis=1, fisher=False, bias=True, nan_policy="omit")

            # 샘플 평균값의 차이(Absolute Error of Means)
            skew_diff = np.abs(np.nanmean(r_skews) - np.nanmean(f_skews))
            kurt_diff = np.abs(np.nanmean(r_kurts) - np.nanmean(f_kurts))

            skew_diffs_assets.append(skew_diff)
            kurt_diffs_assets.append(kurt_diff)

        # 모든 자산에 대한 평균값을 결과 딕셔너리에 저장
        results[f'Skew({w})'] = np.round(np.nanmean(skew_diffs_assets), decimals=4)
        results[f'Kurt({w})'] = np.round(np.nanmean(kurt_diffs_assets), decimals=4)

    # DataFrame 포맷팅 (요청하신 형식)
    ordered_keys = [f"{m}({w})" for m in metrics for w in windows]
    df_avg_scores = pd.DataFrame(
        [results[k] for k in ordered_keys], 
        index=ordered_keys, 
        columns=['Average']
    )
    df_avg_scores.index.name = 'Metric'
    
    return df_avg_scores


def vectorized_batch_acf_3d(data, nlags=30):
    """
    (B, C, T) 형태의 3차원 데이터를 받아 모든 배치와 종목의 ACF를 한 번에 계산
    - data: shape = (B, C, T)
    - return: shape = (B, C, nlags + 1)
    """
    data = np.atleast_3d(data)
    B, C, T = data.shape
    
    # 시간 축(axis=2)을 기준으로 평균 차감
    mu = np.mean(data, axis=2, keepdims=True)
    x = data - mu
    
    # c_0 (분산) - ACF의 분모
    c0 = np.sum(x**2, axis=2) / T
    
    # 결과를 담을 3차원 배열 (lag 0은 항상 1.0)
    acf_res = np.zeros((B, C, nlags + 1))
    acf_res[:, :, 0] = 1.0  
    
    # 시차(lag)별로 행렬 슬라이싱 내적 연산 수행
    for lag in range(1, nlags + 1):
        # 시간 축(axis=2)을 lag만큼 이동시킨 두 배열 간의 내적 (c_k)
        ck = np.sum(x[:, :, :-lag] * x[:, :, lag:], axis=2) / T
        
        # 0으로 나누는 오류 방지
        acf_res[:, :, lag] = np.divide(ck, c0, out=np.zeros((B, C)), where=c0 != 0)
        
    return acf_res


def calculate_acf_score(
    real,
    fake,
    lags=30,
    loss_type='mse'
):
    """
    real: (B_r, C, T)
    fake: (B_f, C, T)
    """
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape

    titles = ['ACF(x)', 'ACF(|x|)', 'ACF(x^2)']
    data_transforms = [lambda x: x, np.abs, np.square]  
    
    # 결과를 저장할 딕셔너리 초기화
    acf_scores = {f"Group {i+1}": {} for i in range(C)}
    avg_scores_per_transform = {title: [] for title in titles}

    for transform, title in zip(data_transforms, titles):
        # (1) 3차원 데이터 전체 변환 (Identity, Absolute, Squared)
        t_real = transform(real)
        t_fake = transform(fake)
        
        # (2) 초고속 3D ACF 병렬 계산 -> shape: (B, C, lags + 1)
        acf_real = vectorized_batch_acf_3d(t_real, nlags=lags)
        acf_fake = vectorized_batch_acf_3d(t_fake, nlags=lags)
        
        # (3) 배치 축(axis=0)에 대한 평균 ACF 도출 -> shape: (C, lags + 1)
        # 배치(B) 사이즈가 서로 달라도 이 단계에서 대푯값으로 압축되므로 문제없이 비교 가능합니다.
        mean_acf_real = np.mean(acf_real, axis=0)
        mean_acf_fake = np.mean(acf_fake, axis=0)
        
        # (4) Real 평균 ACF vs Fake 평균 ACF 오차 계산 -> shape: (C, lags + 1)
        diff = mean_acf_fake - mean_acf_real
        
        # (5) 종목(C)별로 Loss 계산
        for i in range(C):
            # _scalar_diff_loss가 1D 배열(lags + 1 길이)을 받는다고 가정
            val = round(float(_scalar_diff_loss(diff[i], loss_type)), 4)
            acf_scores[f"Group {i+1}"][title] = val
            avg_scores_per_transform[title].append(val)
            
    # (6) 전체 "평균의 평균(Mean of Means)" 계산
    overall_average_scores = {
        title: round(np.mean(vals), 4) for title, vals in avg_scores_per_transform.items()
    }

    # (7) DataFrame 포맷팅
    df_overall = pd.DataFrame(list(overall_average_scores.values()), 
                              index=overall_average_scores.keys(), 
                              columns=['Average'])
    df_overall.index.name = 'Metric'

    return acf_scores, df_overall



def vectorized_batch_leverage_effect_3d(data, lags=30):
    """
    (B, C, T) 형태의 3차원 데이터를 받아 모든 배치와 종목의 레버리지 효과를 한 번에 계산
    - data: shape = (B, C, T)
    - return: shape = (B, C, lags)
    """
    data = np.atleast_3d(data)
    B, C, T = data.shape
    
    # 결과를 담을 3차원 배열
    leverage_res = np.zeros((B, C, lags))
    
    # tau(시차)별로 슬라이싱하여 모든 샘플, 모든 종목의 상관계수를 행렬 내적으로 한 번에 계산
    for tau in range(1, lags + 1):
        # 시간 축(axis=2)을 기준으로 슬라이싱
        # X: r_t (앞부분), Y: r_{t+tau}^2 (뒷부분의 제곱)
        X = data[:, :, :-tau]
        Y = data[:, :, tau:]**2
        
        # 시간 축(axis=2) 기준 평균 차감 (Z-score 분자 부분)
        X_c = X - np.mean(X, axis=2, keepdims=True)
        Y_c = Y - np.mean(Y, axis=2, keepdims=True)
        
        # 상관계수 분자 (공분산 계산 시 분모인 N-1은 약분되므로 생략 가능)
        cov_num = np.sum(X_c * Y_c, axis=2)
        
        # 상관계수 분모 (분산 계산 시 분모인 N-1은 약분됨)
        var_X = np.sum(X_c**2, axis=2)
        var_Y = np.sum(Y_c**2, axis=2)
        denom = np.sqrt(var_X * var_Y)
        
        # 0으로 나누는 오류 방지
        leverage_res[:, :, tau - 1] = np.divide(cov_num, denom, out=np.zeros((B, C)), where=denom != 0)
        
    return leverage_res


def calculate_leverage_effect(real, fake, lags=30, loss_type='mse'):
    """
    real: (B_r, C, T)
    fake: (B_f, C, T)
    """
    real = np.atleast_3d(real)
    fake = np.atleast_3d(fake)
    
    B_r, C, T_r = real.shape
    B_f, _, T_f = fake.shape

    leverage_scores = {}
    all_group_losses = []

    # (1) 3D 벡터화를 통한 초고속 병렬 계산 -> shape: (B_r, C, lags) & (B_f, C, lags)
    real_leverage = vectorized_batch_leverage_effect_3d(real, lags=lags)
    fake_leverage = vectorized_batch_leverage_effect_3d(fake, lags=lags)

    # (2) 배치 축(axis=0)에 대한 평균 도출 -> shape: (C, lags)
    # 배치 크기(B_r, B_f)가 달라도 여기서 종목별 대푯값으로 압축되므로 1:1 비교가 가능해집니다.
    mean_real_leverage = np.mean(real_leverage, axis=0)
    mean_fake_leverage = np.mean(fake_leverage, axis=0)

    # (3) Real과 Fake 평균 간의 차이 계산 -> shape: (C, lags)
    diff = mean_fake_leverage - mean_real_leverage

    # (4) 종목(C)별로 Loss 계산
    for i in range(C):
        # _scalar_diff_loss가 1D 배열(lags 길이)을 받는다고 가정
        loss_val = round(float(_scalar_diff_loss(diff[i], loss_type)), 4)
        leverage_scores[f'Group {i+1}'] = loss_val
        all_group_losses.append(loss_val)

    # (5) 전체 평균 손실 계산
    overall_mean = round(float(np.mean(all_group_losses)), 4)

    # (6) DataFrame 래핑
    df_overall = pd.DataFrame(
        [overall_mean], 
        index=['Leverage Effect'], 
        columns=['Average']
    )
    df_overall.index.name = 'Metric'

    return leverage_scores, df_overall

# ---------------------------------------------------------
# [내부 헬퍼 함수] 시각화 함수 내부에서 쓸 초고속 연산기
# ---------------------------------------------------------
def _fast_acf_2d(data, nlags):
    """시각화용 초고속 2D ACF 계산기"""
    data = np.atleast_2d(data)
    N, T = data.shape
    mu = np.mean(data, axis=1, keepdims=True)
    x = data - mu
    c0 = np.sum(x**2, axis=1) / T
    
    res = np.zeros((N, nlags + 1))
    res[:, 0] = 1.0
    for lag in range(1, nlags + 1):
        ck = np.sum(x[:, :-lag] * x[:, lag:], axis=1) / T
        res[:, lag] = np.divide(ck, c0, out=np.zeros(N), where=c0 != 0)
    return res


In [12]:
real_data = test_data.transpose(1, 2)
# real_data = torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2)
if hasattr(real_data, 'cpu'):
    real_data = real_data.cpu().detach().numpy()
total_result_df=pd.DataFrame()
for seed in range(5):

    """ Fake data와 Test data를 비교 """
    batch_size = 2000
    noise = torch.randn(batch_size, config.noise_dim, config.n_steps, device=config.device)
    with torch.no_grad():
        fake = generator(noise)

    fake_data = inverse_scaling(fake, scalers)
    print(fake_data.shape)
    fake_list = [fake_data[:, i, :] for i in range(fake_data.shape[1])]
    real_list = [real_data[:, i, :] for i in range(real_data.shape[1])]


    """ Visualize the distribution of the real and fake data """

    # Plot the distribution of the real and fake data
    windows = [1, 5, 20, 100]

    result_df=pd.DataFrame()
    CCF_df = calculate_ccf(real_data, fake_data, 0)
    result_df = pd.concat([result_df, CCF_df], axis=0)
    total_CCF_df=pd.DataFrame(columns=['Average'])
    for i in range(1,11):
        total_CCF_df = pd.concat([total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1)

    result_df = pd.concat([result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=['Average'])], axis=0)

    similarity_df = calculate_distribution_scores(real_data, fake_data, windows)
    result_df = pd.concat([result_df, similarity_df], axis=0)
    skew_kurt_df = calculate_skew_kurtosis(real_data, fake_data, windows)
    result_df = pd.concat([result_df, skew_kurt_df], axis=0)
    _, acf_df = calculate_acf_score(real_data, fake_data, 40)
    result_df = pd.concat([result_df, acf_df], axis=0)
    _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
    result_df = pd.concat([result_df, leverage_df], axis=0)

    total_result_df = pd.concat([total_result_df, result_df], axis=1)

statistics_df = pd.DataFrame(index=total_result_df.index, columns=['MEAN', 'STD'])
statistics_df.loc[statistics_df.index, 'MEAN']=total_result_df.mean(axis=1).values
statistics_df.loc[statistics_df.index, 'STD']=total_result_df.std(axis=1).values
print(statistics_df)

(2000, 6, 128)
(2000, 6, 128)
(2000, 6, 128)
(2000, 6, 128)
(2000, 6, 128)
                     MEAN       STD
CCF(τ = 0)_MAE    0.13708  0.000766
CCF(τ = 0)_FROB   0.98904  0.005615
CCF(τ = 0)_MSE    0.02716  0.000329
CCF(τ > 0)_MAE   0.091382  0.000194
CCF(τ > 0)_FROB   1.70183  0.003933
CCF(τ > 0)_MSE   0.014778  0.000076
EMD(1)            0.00182  0.000045
EMD(5)            0.00586  0.000055
EMD(20)           0.01466  0.000055
EMD(100)          0.04422  0.000295
KS(1)             0.05016  0.000404
KS(5)             0.07274   0.00023
KS(20)            0.11094  0.000966
KS(100)            0.1913  0.001651
Skew(1)           0.24542  0.007718
Skew(5)           0.22574  0.008537
Skew(20)          0.18536  0.001721
Skew(100)         0.02912  0.004206
Kurt(1)           1.16496  0.044197
Kurt(5)           0.53534  0.010439
Kurt(20)           0.2253  0.002796
Kurt(100)          0.0874  0.007409
ACF(x)             0.0026       0.0
ACF(|x|)          0.00202  0.000045
ACF(x^2)          0.00122